# Analysis Notebook

**Framework-Grounded Comparative Evaluation of TA vs Student Simulation Protocols**

Run this notebook after:
- P6 main generation complete
- P7 trace metrics computed
- P8 coder ratings ingested via `src/coding/ingest_ratings.py`

Statistical plan is pre-registered internally before viewing condition results.
Primary comparisons: C1 vs C2, C1 vs C3, C2 vs C3, {C1,C2} vs C4.
Bonferroni α = 0.05 / 4 = 0.0125.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')

## 1. Load All Data

In [ ]:
ratings = pd.read_csv('../outputs/coder_ratings_raw.csv') if Path('../outputs/coder_ratings_raw.csv').exists() else pd.DataFrame()
trace = pd.read_csv('../outputs/automatic_trace_metrics.csv') if Path('../outputs/automatic_trace_metrics.csv').exists() else pd.DataFrame()
scenarios = pd.read_csv('../data/scenarios.csv') if Path('../data/scenarios.csv').exists() else pd.DataFrame()
reliability = pd.read_csv('../outputs/reliability_report.csv') if Path('../outputs/reliability_report.csv').exists() else pd.DataFrame()

print(f'Ratings: {ratings.shape}')
print(f'Trace metrics: {trace.shape}')
print(f'Scenarios: {scenarios.shape}')

## 2. Sample Size & Exclusion Report

In [ ]:
gen_dialogues = pd.read_csv('../outputs/generated_dialogues.csv') if Path('../outputs/generated_dialogues.csv').exists() else pd.DataFrame()
if not gen_dialogues.empty:
    main = gen_dialogues[gen_dialogues['phase']=='main']
    print('=== Generation Summary ===')
    print(f'Total: {len(main)}')
    print(f'Valid: {(~main["exclusion_flag"]).sum()} ({100*(~main["exclusion_flag"]).mean():.1f}%)')
    print('\nExclusion rate by condition:')
    print(main.groupby('condition')['exclusion_flag'].mean().round(3))
    print('\nExclusion codes:')
    print(main[main['exclusion_flag']]['exclusion_code'].value_counts())

## 3. Inter-Rater Reliability

In [ ]:
if not reliability.empty:
    print('=== Reliability Report ===')
    print('Go criterion: ICC ≥ 0.6 or α ≥ 0.6')
    display_cols = ['dimension','icc_2k','krippendorff_alpha','go_criterion_met','note']
    print(reliability[display_cols].to_string(index=False))

## 4. Primary Confirmatory Analysis

### 4a. TA Dimensions (C1 expected leader)

In [ ]:
from src.analysis.mixed_models import run_analysis
run_analysis()

In [ ]:
mm_results = pd.read_csv('../outputs/mixed_model_results.csv') if Path('../outputs/mixed_model_results.csv').exists() else pd.DataFrame()

if not ratings.empty:
    ta_dims = [d for d in ['TA1','TA2','TA3','TA4'] if d in ratings.columns]
    resolved = ratings.groupby(['packet_id','true_condition','scenario_id'])[ta_dims+[d for d in ['SS1','SS2','SS3','SS4','SS5'] if d in ratings.columns]].mean().reset_index()

    if ta_dims:
        fig, axes = plt.subplots(1, len(ta_dims), figsize=(4*len(ta_dims), 4), sharey=False)
        if len(ta_dims)==1: axes=[axes]
        cond_order = ['C1','C2','C3','C4']
        for ax, dim in zip(axes, ta_dims):
            sns.boxplot(data=resolved, x='true_condition', y=dim, order=cond_order, ax=ax)
            ax.set_title(f'{dim}\n(Expected: C1 highest)')
            ax.set_xlabel('Condition'); ax.set_ylim(0.5, 5.5)
        plt.suptitle('TA Dimensions by Condition (Blair et al., 2007)', fontsize=12, y=1.02)
        plt.tight_layout()
        plt.savefig('../outputs/ta_dimensions_boxplot.png', bbox_inches='tight')
        plt.show()

### 4b. SS Dimensions (C2 expected leader)

In [ ]:
if not ratings.empty:
    ss_dims = [d for d in ['SS1','SS2','SS3','SS4','SS5'] if d in ratings.columns]
    if ss_dims:
        fig, axes = plt.subplots(1, len(ss_dims), figsize=(4*len(ss_dims), 4), sharey=False)
        if len(ss_dims)==1: axes=[axes]
        for ax, dim in zip(axes, ss_dims):
            sns.boxplot(data=resolved, x='true_condition', y=dim, order=cond_order, ax=ax)
            ax.set_title(f'{dim}\n(Expected: C2 highest)')
            ax.set_xlabel('Condition'); ax.set_ylim(0.5, 5.5)
        plt.suptitle('SS Dimensions by Condition (Koedinger et al., 2015)', fontsize=12, y=1.02)
        plt.tight_layout()
        plt.savefig('../outputs/ss_dimensions_boxplot.png', bbox_inches='tight')
        plt.show()

### 4c. Effect Size Summary (Cohen's d)

In [ ]:
if not mm_results.empty:
    primary_comps = ['C1_vs_C2','C1_vs_C3','C2_vs_C3','C1_vs_C4','C2_vs_C4']
    primary = mm_results[mm_results['comparison'].isin(primary_comps)]
    pivot_d = primary.pivot_table(index='dimension', columns='comparison', values='cohens_d')
    print('Cohen d (primary comparisons):')
    print(pivot_d.round(2).to_string())
    
    # Heatmap
    fig, ax = plt.subplots(figsize=(8,4))
    sns.heatmap(pivot_d.astype(float), annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
    ax.set_title("Cohen's d by Dimension × Comparison (positive = row condition higher)")
    plt.tight_layout()
    plt.savefig('../outputs/effect_size_heatmap.png', bbox_inches='tight')
    plt.show()

## 5. Binary Trace Metrics

In [ ]:
from src.analysis.logistic_models import run_analysis as run_logistic
run_logistic()

if not trace.empty:
    key_metrics = ['rule_question_asking_rate','rule_near_transfer_attempt',
                   'rule_target_error_preservation','rule_premature_correctness','rule_any_role_drift']
    avail = [m for m in key_metrics if m in trace.columns]
    if avail:
        props = trace.groupby('condition')[avail].mean().round(3)
        props.T.plot(kind='bar', figsize=(10,4))
        plt.title('Key Trace Metrics by Condition')
        plt.ylabel('Proportion')
        plt.xticks(rotation=45, ha='right')
        plt.legend(title='Condition')
        plt.tight_layout()
        plt.savefig('../outputs/trace_metrics_barplot.png', bbox_inches='tight')
        plt.show()

## 6. Use-Case Decision Analysis

In [ ]:
from src.analysis.use_case_confusion import run_analysis as run_uc
run_uc()

cm_path = Path('../outputs/use_case_confusion_matrix.csv')
if cm_path.exists():
    cm = pd.read_csv(cm_path, index_col=0)
    fig, ax = plt.subplots(figsize=(7,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Assigned Use Case'); ax.set_ylabel('Actual Condition')
    ax.set_title('Use-Case Decision Confusion Matrix\n(diagonal = correct assignment)')
    plt.tight_layout()
    plt.savefig('../outputs/use_case_confusion_heatmap.png', bbox_inches='tight')
    plt.show()

## 7. Robustness Analysis

In [ ]:
from src.analysis.robustness import run_analysis as run_robust
run_robust()

robust_path = Path('../outputs/robustness_results.csv')
if robust_path.exists():
    rob = pd.read_csv(robust_path)
    # Consistency: % splits where C1>C2 on TA dims, C2>C3 on SS dims
    for dim_group, comp in [(['TA1','TA2','TA3','TA4'],'C1_vs_C2'), (['SS1','SS2','SS3','SS4','SS5'],'C2_vs_C3')]:
        subset = rob[rob['dimension'].isin(dim_group) & (rob['comparison']==comp)]
        if not subset.empty:
            direction_counts = subset['direction'].value_counts(normalize=True)
            print(f'{comp} on {dim_group[0]}–{dim_group[-1]}: {direction_counts.to_dict()}')

## 8. Public-Reference Similarity

In [ ]:
from src.analysis.reference_similarity import run_analysis as run_ref
run_ref()

sim_path = Path('../outputs/baseline_similarity_table.csv')
if sim_path.exists():
    sim = pd.read_csv(sim_path)
    print('NOTE: Similarity/divergence from MathDial public reference only.')
    print('This does NOT constitute a realism claim.')
    print()
    print(sim[['feature','condition','generated_mean','reference_mean','divergence']].to_string(index=False))

## 9. Claim Boundary Check

Before writing the paper, verify the following:

| Allowed | Evidence Required |
|---------|-------------------|
| C1 vs C2 produce different evidence | P9 mixed model results |
| C1 better supports TA use (conditional) | C1 sig > C3,C4 on TA1–TA4 with d ≥ 0.3 |
| C2 better supports SS use (conditional) | C2 sig > C3,C4 on SS1–SS5 with d ≥ 0.3 |

| Forbidden | Reason |
|-----------|--------|
| Human learning gains | No classroom data collected |
| Real-student realism | MathDial is reference, not ground truth |
| Validated psychometric scales | No scale validation performed |

In [ ]:
print('Analysis pipeline complete.')
print('All outputs saved to ../outputs/')